# Таблицы, преобразования и качество данных

Модуль 1. Обработка данных в Pandas. Ноутбук 1 из 4.

Данные интернет-магазина используются для построения проверенной таблицы заказов. Последовательность включает загрузку, выборку, преобразование типов, обработку дефектов и соединение справочников. Объяснения сопровождают выполненные примеры; задачи и их решения входят в общую последовательность.

Ноутбук независим от остальных файлов комплекта. Исходные CSV находятся в `course/data`; рабочая папка ядра обычно совпадает с папкой ноутбука. Поиск данных выполняется только по ожидаемому пути `data/orders.csv` относительно текущей папки и её родителей. Данные не загружаются из сети и исходные файлы не изменяются.

Все импорты сосредоточены в первой кодовой ячейке. Сохранённые результаты показывают фактические вычисления на предоставленных данных, а не вручную подготовленные ответы. Даты относятся к синтетическому учебному набору за 2026 год; календарь набора не равен текущему моменту наблюдения.

## 1. Устройство набора данных и таблицы в памяти


Перед вычислениями определяется **зерно таблицы**: смысл одной строки. В этом наборе строка `orders.csv` описывает один заказ с одним товаром, а `order_id` уникален. В реальной системе одна строка может быть позицией заказа; тогда количество строк и количество заказов уже различаются. Уникальность в учебном файле проверяется, а не переносится на любые другие данные.

`customers.csv` содержит сведения о покупателях, `products.csv` — о товарах. Идентификатор связывает сущности, но не является количественной характеристикой: сумма идентификаторов товаров не имеет аналитического смысла. Основные структуры Pandas — одномерная `Series` и двумерный `DataFrame`; обе имеют индекс, который участвует в адресации и выравнивании.

`Path` представляет путь как объект и позволяет строить его без ручного соединения разделителей. Стандартные библиотеки Python используются для путей, NumPy — для числовых операций, Pandas — для таблиц. `display` сохраняет табличное представление нескольких результатов одной ячейки.

Поиск каталога данных не перебирает содержимое проекта: проверяется только ожидаемый файл `data/orders.csv` у текущей папки и её родителей. При отсутствии файла возникает понятная ошибка. Параметры отображения ограничивают объём вывода, но не удаляют строки из самой таблицы. Версии библиотек фиксируют среду, в которой получены сохранённые результаты.

In [1]:
# Шаг 01.01 — Среда и поиск исходных файлов
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 16)
pd.set_option('display.max_rows', 12)
pd.set_option('display.width', 140)
data_candidates = [parent / 'data' for parent in (Path.cwd(), *Path.cwd().parents)]
DATA_DIR = next((candidate for candidate in data_candidates if (candidate / 'orders.csv').is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Не найден course/data/orders.csv относительно рабочей папки ядра')
display(pd.DataFrame({'компонент': ['Pandas', 'NumPy', 'данные'],
                      'значение': [pd.__version__, np.__version__, str(DATA_DIR)]}))

,компонент,значение
0,Pandas,2.3.3
1,NumPy,2.4.6
2,данные,C:\Users\user\Downloads\BigData_Kit\course\data


`read_csv` читает файл и создаёт `DataFrame`. В первой загрузке даты намеренно остаются строками: это позволяет увидеть разницу между текстовым и календарным представлением. Наличие цифр в строке ещё не означает подходящий тип.

Объём и перечень столбцов составляют минимальный паспорт входа. Эти сведения помогают заметить неправильный разделитель, отсутствие заголовка или чтение другого файла. В каталоге `data/hive` находятся варианты для другого способа загрузки; здесь используются CSV с заголовками из самого `data`.

In [2]:
# Шаг 01.02 — Загрузка заказов и справочников
orders = pd.read_csv(DATA_DIR / 'orders.csv')
customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
tables = {'заказы': orders, 'покупатели': customers, 'товары': products}
display(pd.DataFrame([
    {'таблица': name, 'строк': len(table), 'столбцов': table.shape[1],
     'поля': ', '.join(table.columns)}
    for name, table in tables.items()
]))

,таблица,строк,столбцов,поля
0,заказы,200000,10,"order_id, order_ts, customer_id, city, product..."
1,покупатели,20000,4,"customer_id, city, registered, segment"
2,товары,2000,4,"product_id, category, name, price"


`head` показывает небольшой фрагмент, `nunique` считает различные непустые значения, а `duplicated` обнаруживает повторы. Проверка ключа включает и уникальность, и отсутствие пропусков: несколько пустых идентификаторов нельзя считать полноценными разными сущностями.

Поля `price` и `amount` выражены в условных денежных единицах набора. По имеющимся полям нельзя восстановить скидки, налоги или реальные платежи. Статус описывает состояние записи в синтетическом снимке; статус и сумма требуют совместного толкования, прежде чем показатель будет назван продажами.

In [3]:
# Шаг 01.03 — Зерно таблицы и уникальность ключей
display(orders.head(6))
key_checks = []
for table_name, key in [('заказы', 'order_id'), ('покупатели', 'customer_id'), ('товары', 'product_id')]:
    frame = tables[table_name]
    key_checks.append({'таблица': table_name, 'ключ': key,
                       'уникален': frame[key].is_unique,
                       'пустых ключей': int(frame[key].isna().sum()),
                       'различных ключей': frame[key].nunique()})
display(pd.DataFrame(key_checks))

,order_id,order_ts,customer_id,city,product_id,category,qty,price,amount,status
0,1,2026-01-17 08:50:04,8384,Санкт-Петербург,1104,Одежда,1,9700,9700,delivered
1,2,2026-07-13 14:51:05,11971,Краснодар,1471,Продукты,1,480,480,delivered
2,3,2026-11-03 13:42:32,19274,Новосибирск,887,Обувь,1,7630,7630,delivered
3,4,2026-03-19 16:38:33,14821,Москва,79,Одежда,1,11660,11660,returned
4,5,2026-11-27 01:54:49,6548,Москва,1397,Бытовая техника,1,57730,57730,delivered
5,6,2026-07-20 15:42:07,17140,Новосибирск,960,Одежда,1,3530,3530,delivered


,таблица,ключ,уникален,пустых ключей,различных ключей
0,заказы,order_id,True,0,200000
1,покупатели,customer_id,True,0,20000
2,товары,product_id,True,0,2000


Один выбранный столбец обычно возвращает `Series`, список столбцов — `DataFrame`. Индекс — метки строк; он не обязан быть последовательностью от нуля и не становится бизнес-ключом автоматически.

`loc` выбирает по меткам, `iloc` — по позициям. Срез `loc` по меткам включает правую границу, а позиционный срез `iloc` следует обычному правилу Python и её исключает. После установки `customer_id` индексом метка 1 означает покупателя с этим идентификатором, а не вторую строку. Установка индекса не сортирует данные и не проверяет уникальность без явного параметра.

In [4]:
# Шаг 01.04 — Series DataFrame и адресация
display(pd.DataFrame({'выражение': ["orders['amount']", "orders[['amount']]"],
                      'тип': [type(orders['amount']).__name__, type(orders[['amount']]).__name__]}))
customer_index = customers.set_index('customer_id', verify_integrity=True)
display(customer_index.loc[[1, 2], ['city', 'segment']])
display(customer_index.iloc[:2, :2])

,выражение,тип
0,orders['amount'],Series
1,orders[['amount']],DataFrame


,city,segment
customer_id,,
1,Москва,regular
2,Москва,regular


,city,registered
customer_id,,
1,Москва,2022-09-01
2,Москва,2024-11-30


`info` объединяет сведения о типах, заполненности и приблизительном объёме памяти. `describe` даёт статистическое описание числовых столбцов. Числа в идентификаторе не делают средний идентификатор полезным показателем, поэтому для анализа выбираются только содержательные количественные поля.

`value_counts(dropna=False)` показывает частоты категорий вместе с пустыми значениями. Полнота столбца не доказывает корректность: отрицательная цена тоже является заполненной ячейкой. Поиск пропусков, статистика значений и проверка бизнес-правил решают разные задачи.

In [5]:
# Шаг 01.05 — Профиль типов и значений
orders.info(memory_usage='deep')
display(orders[['qty', 'price', 'amount']].describe().round(2))
display(orders['status'].value_counts(dropna=False).rename_axis('status').to_frame('заказов'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   order_id     200000 non-null  int64 
 1   order_ts     200000 non-null  object
 2   customer_id  200000 non-null  int64 
 3   city         200000 non-null  object
 4   product_id   200000 non-null  int64 
 5   category     200000 non-null  object
 6   qty          200000 non-null  int64 
 7   price        200000 non-null  int64 
 8   amount       200000 non-null  int64 
 9   status       200000 non-null  object
dtypes: int64(6), object(4)
memory usage: 64.6 MB


,qty,price,amount
count,200000.00,200000.00,200000.00
mean,1.49,21032.33,31332.10
std,0.90,28813.10,53568.49
min,1.00,90.00,90.00
25%,1.00,2470.00,3040.00
50%,1.00,8170.00,10430.00
75%,2.00,24140.00,34910.00
max,5.00,119880.00,599400.00


,заказов
status,
delivered,164061
cancelled,17980
returned,10045
in_transit,7914


**Промежуточный результат.** Определены размеры таблиц, смысл строк, ключи и базовые типы. Исходный набор загружен без изменения файлов. Следующий этап — выразить условия отбора и подготовить значения для вычислений.

## 2. Выборка и создание признаков


Фильтрация строится как логическое условие над столбцами. Результат сравнения — булева `Series`, которая выбирает строки по совпадающим индексам. Условия объединяются поэлементными `&`, `|`, `~`; каждое составное сравнение заключается в скобки.

Создание признака должно сохранять понятный смысл строки. Цена, диапазон суммы и календарный месяц описывают текущий заказ, а сводные характеристики покупателей потребуют отдельной агрегации. Здесь рассматриваются преобразования внутри строки; группировки подробно используются в следующем ноутбуке.

`isin` проверяет принадлежность набору, `between` — попадание в интервал. Скобки отделяют сравнения до применения поэлементного логического оператора. Обычные `and` и `or` работают с одиночными логическими значениями Python и не заменяют `&` и `|` для столбцов.

В примере отбираются доставленные заказы из двух городов с суммой от 5 000 до 20 000 включительно. Поля результата заданы явно: это одновременно уменьшает визуальный шум и делает смысл выборки понятным. Количество найденных записей проверяется отдельно от нескольких показанных строк.

In [6]:
# Шаг 01.06 — Логические условия и выбор столбцов
mask = (orders['status'].eq('delivered')
        & orders['city'].isin(['Москва', 'Санкт-Петербург'])
        & orders['amount'].between(5_000, 20_000, inclusive='both'))
selected_columns = ['order_id', 'city', 'category', 'qty', 'amount', 'status']
display(orders.loc[mask, selected_columns].head(8))
print('Всего подходящих заказов:', int(mask.sum()))

,order_id,city,category,qty,amount,status
0,1,Санкт-Петербург,Одежда,1,9700,delivered
15,16,Москва,Одежда,1,5150,delivered
19,20,Москва,Обувь,2,6280,delivered
24,25,Санкт-Петербург,Красота,1,6400,delivered
25,26,Москва,Красота,1,7510,delivered
49,50,Москва,Спорт,4,6640,delivered
53,54,Москва,Одежда,1,12140,delivered
54,55,Москва,Обувь,1,15060,delivered


Всего подходящих заказов: 26541


Подготовка отдельной выборки не должна неявно менять исходную таблицу. Для независимой работы используется `.copy()`, затем присваивание через `.loc`. Цепочки вида `frame[condition]['column'] = value` создают неоднозначность и не являются надёжным способом изменения исходного объекта.

Новый столбец `контроль` существует только в копии выбранных заказов. Сравнение наличия столбца в двух объектах подтверждает границу изменений. Это особенно важно в ноутбуке: состояние объектов сохраняется между ячейками, поэтому случайное изменение исходной таблицы влияет на последующие результаты.

In [7]:
# Шаг 01.07 — Присваивание в явную копию
selected = orders.loc[mask, selected_columns].copy()
selected.loc[:, 'контроль'] = 'обычный'
selected.loc[selected['amount'].ge(15_000), 'контроль'] = 'повышенное внимание'
display(selected.head(6))
print('Поле есть в копии:', 'контроль' in selected.columns)
print('Поле есть в исходной таблице:', 'контроль' in orders.columns)

,order_id,city,category,qty,amount,status,контроль
0,1,Санкт-Петербург,Одежда,1,9700,delivered,обычный
15,16,Москва,Одежда,1,5150,delivered,обычный
19,20,Москва,Обувь,2,6280,delivered,обычный
24,25,Санкт-Петербург,Красота,1,6400,delivered,обычный
25,26,Москва,Красота,1,7510,delivered,обычный
49,50,Москва,Спорт,4,6640,delivered,обычный


Поле есть в копии: True
Поле есть в исходной таблице: False


`sort_values` сортирует по значениям одного или нескольких столбцов. Направления задаются отдельно для каждого ключа. Дополнительный ключ делает порядок однозначным при одинаковых суммах: сначала сумма по убыванию, затем идентификатор по возрастанию.

`head(8)` после сортировки означает восемь первых строк в полученном порядке. Это не восемь различных сумм и не восемь товаров: зерно остаётся заказом. Возврат новой таблицы без `inplace=True` упрощает отслеживание состояния исходных данных.

In [8]:
# Шаг 01.08 — Сортировка и устойчивый порядок результата
largest_orders = orders.sort_values(['amount', 'order_id'], ascending=[False, True])
display(largest_orders[['order_id', 'category', 'qty', 'price', 'amount']].head(8))

,order_id,category,qty,price,amount
96989,96990,Электроника,5,119880,599400
110653,110654,Электроника,5,119790,598950
10231,10232,Электроника,5,119340,596700
63201,63202,Электроника,5,119340,596700
147706,147707,Электроника,5,118980,594900
148003,148004,Электроника,5,118980,594900
23914,23915,Электроника,5,118470,592350
43241,43242,Электроника,5,118470,592350


Арифметика над столбцами выполняется поэлементно. Проверка `amount == qty * price` подтверждает согласованность трёх полей, но не доказывает правильность самой цены с точки зрения внешнего источника.

`map` по словарю заменяет каждый код соответствующим значением. Отсутствующий ключ даёт пропуск; он полезен как сигнал неизвестной категории. Словарь перевода не меняет исходный смысл статуса. Повторное присваивание описаний в отдельный столбец сохраняет машинный код для проверок и фильтрации.

In [9]:
# Шаг 01.09 — Векторные вычисления и map
status_names = {'delivered': 'доставлен', 'returned': 'возвращён',
                'cancelled': 'отменён', 'in_transit': 'в пути'}
orders = orders.assign(calculated_amount=orders['qty'] * orders['price'],
                       status_name=orders['status'].map(status_names))
display(orders[['order_id', 'qty', 'price', 'amount', 'calculated_amount', 'status_name']].head(6))
print('Расхождений суммы:', int(orders['amount'].ne(orders['calculated_amount']).sum()))
print('Неизвестных статусов:', int(orders['status_name'].isna().sum()))

,order_id,qty,price,amount,calculated_amount,status_name
0,1,1,9700,9700,9700,доставлен
1,2,1,480,480,480,доставлен
2,3,1,7630,7630,7630,доставлен
3,4,1,11660,11660,11660,возвращён
4,5,1,57730,57730,57730,доставлен
5,6,1,3530,3530,3530,доставлен


Расхождений суммы: 0
Неизвестных статусов: 0


`where` оставляет значение там, где условие истинно, и использует замену в остальных строках. В примере недоставленным заказам назначается нулевой вклад в показатель стоимости доставленных заказов. Это определение показателя, а не исправление исходной суммы.

`cut` использует заданные границы, поэтому одинаковая сумма всегда относится к одному диапазону. `qcut` подбирает границы по распределению и может столкнуться с одинаковыми границами при большом числе повторов. Интервалы имеют включённую и исключённую стороны; в данном примере правая граница включена. Бизнес-пороги и квантили отвечают на разные вопросы.

In [10]:
# Шаг 01.10 — Условные значения where и интервалы cut
orders['delivered_amount'] = orders['amount'].where(orders['status'].eq('delivered'), 0)
orders['amount_band'] = pd.cut(orders['amount'], bins=[0, 5_000, 20_000, 100_000, np.inf],
                               labels=['до 5 тыс.', '5–20 тыс.', '20–100 тыс.', 'свыше 100 тыс.'])
quantile_example = pd.qcut(orders['amount'], q=4, duplicates='drop')
display(orders[['order_id', 'amount', 'status', 'delivered_amount', 'amount_band']].head(8))
display(quantile_example.value_counts(sort=False).to_frame('заказов'))

,order_id,amount,status,delivered_amount,amount_band
0,1,9700,delivered,9700,5–20 тыс.
1,2,480,delivered,480,до 5 тыс.
2,3,7630,delivered,7630,5–20 тыс.
3,4,11660,returned,0,5–20 тыс.
4,5,57730,delivered,57730,20–100 тыс.
5,6,3530,delivered,3530,до 5 тыс.
6,7,5350,delivered,5350,5–20 тыс.
7,8,119270,delivered,119270,свыше 100 тыс.


,заказов
amount,
"(89.999, 3040.0]",50041
"(3040.0, 10430.0]",50014
"(10430.0, 34910.0]",49978
"(34910.0, 599400.0]",49967


Календарный тип позволяет извлекать компоненты даты и рассчитывать интервалы. `to_datetime` с явным форматом проверяет ожидаемую запись, а `errors='raise'` прекращает обработку при нарушении. Время набора не содержит часового пояса, поэтому его нельзя без дополнительных сведений считать UTC.

Месяц как `Period` обозначает календарный интервал, а не строковую подпись. День недели в Pandas нумеруется от нуля для понедельника. Вычитание дат даёт `Timedelta`; полные дни и календарные даты следует различать. Агрегации по периодам и временные ряды подробно рассматриваются в аналитическом ноутбуке.

In [11]:
# Шаг 01.11 — Календарные типы и длительности
orders['order_ts'] = pd.to_datetime(orders['order_ts'], format='%Y-%m-%d %H:%M:%S', errors='raise')
customers['registered'] = pd.to_datetime(customers['registered'], format='%Y-%m-%d', errors='raise')
orders['order_month'] = orders['order_ts'].dt.to_period('M')
orders['weekday'] = orders['order_ts'].dt.dayofweek
display(orders[['order_id', 'order_ts', 'order_month', 'weekday']].head(6))
print('Период набора:', orders['order_ts'].min(), '—', orders['order_ts'].max())

,order_id,order_ts,order_month,weekday
0,1,2026-01-17 08:50:04,2026-01,5
1,2,2026-07-13 14:51:05,2026-07,0
2,3,2026-11-03 13:42:32,2026-11,1
3,4,2026-03-19 16:38:33,2026-03,3
4,5,2026-11-27 01:54:49,2026-11,4
5,6,2026-07-20 15:42:07,2026-07,0


Период набора: 2026-01-01 00:00:19 — 2026-12-31 23:58:45


Строковые методы `.str` применяются к элементам столбца, а не к имени столбца. Удаление лишних пробелов, нормализация регистра и согласование сокращений решают разные задачи. Нормализация не должна превращаться в угадывание неизвестных значений.

Тип `string` сохраняет семантику текстового столбца и специальное пустое значение `pd.NA`. Приведение через `astype(str)` превратило бы некоторые пропуски в буквальные строки, что затрудняет проверку. `replace` выполняет явные замены, а регулярное выражение применяется только там, где правило действительно нужно.

In [12]:
# Шаг 01.12 — Строковая нормализация и replace
city_example = pd.Series(['  г. Москва ', 'МОСКВА', 'Санкт-Петербург', None], dtype='string')
city_normalized = (city_example.str.strip().str.replace(r'^г\.\s*', '', regex=True)
                   .str.casefold().replace({'москва': 'Москва', 'санкт-петербург': 'Санкт-Петербург'}))
display(pd.DataFrame({'исходное значение': city_example, 'нормализованное': city_normalized}))

,исходное значение,нормализованное
0,г. Москва,Москва
1,МОСКВА,Москва
2,Санкт-Петербург,Санкт-Петербург
3,<NA>,<NA>


**Промежуточный результат.** Условия отбора выражены явно, признаки вычисляются над столбцами, даты представлены календарным типом. Изменения рабочей копии отделены от исходных данных. Далее те же операции применяются к воспроизводимому набору дефектов.

## 3. Очистка как проверяемая работа с дефектами


Предоставленные CSV не содержат пустых ячеек, повторов первичных ключей и нарушений арифметического равенства суммы. Для практики создаётся небольшая копия с контролируемыми дефектами. Исходный набор остаётся эталоном сравнения и не перезаписывается.

Очистка разделяется на нормализацию записи, выявление нарушений и решение о допустимости строки. **Карантин** означает отдельную таблицу отклонённых записей с причинами; он позволяет не терять след ошибок. Непонятное значение не исправляется произвольным средним или самым частым вариантом только ради отсутствия пропусков.

Ошибки добавляются в разные поля и относятся к разным классам: пропуски, нарушение диапазона, арифметика, неизвестная категория, отсутствующая связь и конфликт ключа. Полностью совпадающая строка и две разные строки с одинаковым идентификатором требуют разной обработки.

Количество представлено nullable-типом `Int64`, чтобы целые значения могли сосуществовать с пропуском. Это имя отличается от NumPy-типа `int64`. Текстовые дефекты пока сохраняются буквально; их нормализация будет отдельным наблюдаемым шагом.

In [13]:
# Шаг 01.13 — Создание контролируемых дефектов
source_head = orders.head(20).copy(deep=True)
dirty = source_head.copy(deep=True)
dirty['qty'] = dirty['qty'].astype('Int64')
dirty.loc[0, 'qty'] = pd.NA
dirty.loc[1, 'price'] = -100
dirty.loc[2, 'amount'] += 7
dirty.loc[3, 'city'] = '  мОсКва '
dirty.loc[4, 'status'] = 'DELIVERED '
dirty.loc[5, 'customer_id'] = 999_999
dirty.loc[6, 'order_ts'] = pd.NaT
dirty.loc[7, 'qty'] = 0
dirty.loc[8, 'status'] = 'unknown'
dirty.loc[9, 'product_id'] = 999_999
dirty.loc[10, 'order_id'] = dirty.loc[11, 'order_id']
dirty = pd.concat([dirty, dirty.iloc[[2]]], ignore_index=True)
display(dirty[['order_id', 'customer_id', 'product_id', 'city', 'qty', 'price', 'amount', 'status']].head(12))
print('Строк в исходном фрагменте:', len(source_head), 'в копии с дефектами:', len(dirty))

,order_id,customer_id,product_id,city,qty,price,amount,status
0,1,8384,1104,Санкт-Петербург,<NA>,9700,9700,delivered
1,2,11971,1471,Краснодар,1,-100,480,delivered
2,3,19274,887,Новосибирск,1,7630,7637,delivered
3,4,14821,79,мОсКва,1,11660,11660,returned
4,5,6548,1397,Москва,1,57730,57730,DELIVERED
5,6,999999,960,Новосибирск,1,3530,3530,delivered
6,7,18461,1568,Нижний Новгород,1,5350,5350,delivered
7,8,6548,1977,Москва,0,119270,119270,delivered
8,9,11148,774,Воронеж,3,2940,8820,unknown
9,10,12,999999,Казань,2,33170,66340,delivered


Строк в исходном фрагменте: 20 в копии с дефектами: 21


`isna` распознаёт специальные пустые значения, в том числе `pd.NA` и `NaT`. Пустая строка и строка из пробелов не обязательно считаются пропуском, поэтому для текстовых полей требуется отдельная договорённость.

У `duplicated` можно задать столбцы и правило маркировки. `keep=False` отмечает все строки конфликтующей группы, а значение по умолчанию отмечает только повторы после первой строки. Число отмеченных строк не равно числу групп и не равно числу удаляемых записей. Диагностический отчёт должен явно называть, что именно посчитано.

In [14]:
# Шаг 01.14 — Профиль пропусков и повторов
missing_profile = dirty.isna().sum().loc[lambda s: s.gt(0)].to_frame('пустых значений')
display(missing_profile)
display(pd.DataFrame({
    'проверка': ['полных повторов после первой строки', 'строк с повторяющимся order_id'],
    'количество': [int(dirty.duplicated().sum()), int(dirty.duplicated('order_id', keep=False).sum())]
}))
display(dirty.loc[dirty.duplicated('order_id', keep=False), ['order_id', 'customer_id', 'product_id', 'amount']])

,пустых значений
order_ts,1
qty,1


,проверка,количество
0,полных повторов после первой строки,1
1,строк с повторяющимся order_id,4


,order_id,customer_id,product_id,amount
2,3,19274,887,7637
10,12,16192,322,87190
11,12,6548,1473,1830
20,3,19274,887,7637


Полный повтор удаляется до проверки уникальности ключа: после этого остаются только содержательные конфликты. Для статуса безопасно убрать внешние пробелы и привести регистр в рамках известного словаря. Для города используется справочник допустимых названий.

Нормализация не исправляет отрицательную цену и не восстанавливает отсутствующий товар. Её задача — привести разные записи одного известного значения к общей форме. После операции всё равно проверяется попадание в допустимое множество. Неизвестные значения остаются заметными.

In [15]:
# Шаг 01.15 — Нормализация известных вариантов
working = dirty.drop_duplicates().copy()
working['status'] = working['status'].astype('string').str.strip().str.casefold()
city_dictionary = dict(zip(customers['city'].str.strip().str.casefold(), customers['city']))
working['city'] = working['city'].astype('string').str.strip().str.casefold().map(city_dictionary)
display(working.loc[[3, 4, 8], ['order_id', 'city', 'status']])
print('После удаления полных повторов:', len(working))

,order_id,city,status
3,4,Москва,returned
4,5,Москва,delivered
8,9,Воронеж,unknown


После удаления полных повторов: 20


`to_numeric(errors='coerce')` заменяет непереводимый текст пропуском. Это не доказательство правильности остальных значений: отрицательное число успешно преобразуется и затем должно проверяться по диапазону. Преобразование сохраняется вместе с маской неудачных непустых значений, чтобы причина пропуска не исчезла.

Nullable-тип `Int64` удобен, если данные действительно целочисленные. Для дробных величин принудительное приведение к целому без правила округления недопустимо. В задачах с деньгами тип и единица измерения выбираются явно, а не выводятся из нескольких первых строк.

In [16]:
# Шаг 01.16 — Безопасное преобразование ошибочных чисел
raw_numbers = pd.Series(['100', ' 250 ', 'ошибка', None, '-5'], dtype='string')
numeric_values = pd.to_numeric(raw_numbers, errors='coerce').astype('Int64')
conversion_failed = raw_numbers.notna() & numeric_values.isna()
display(pd.DataFrame({'исходное': raw_numbers, 'число': numeric_values,
                      'ошибка преобразования': conversion_failed,
                      'нарушение положительности': numeric_values.le(0).fillna(False)}))

,исходное,число,ошибка преобразования,нарушение положительности
0,100,100,False,False
1,250,250,False,False
2,ошибка,<NA>,True,False
3,<NA>,<NA>,False,False
4,-5,-5,False,True


`fillna` задаёт замену, а `dropna(subset=...)` удаляет строки только по явно названным обязательным полям. Это разные решения: заполнение меняет значение, удаление меняет состав наблюдаемой совокупности. Ни одно не является универсальной очисткой.

В отдельном примере неизвестный канал обращения получает явную категорию «не указан». Пропуск числа обращений остаётся неизвестным: ноль означал бы доказанное отсутствие обращений. Запись без обязательного идентификатора отделяется. Заполнение средней или медианой может изменить распределение и исказить связи, поэтому не применяется механически. Пустоты количества товара в заказе не заполняются таким способом.

In [17]:
# Шаг 01.17 — Заполнение и удаление пропусков по смыслу
contact_example = pd.DataFrame({
    'customer_id': pd.Series([1, 2, pd.NA, 4], dtype='Int64'),
    'channel': pd.Series(['сайт', pd.NA, 'телефон', pd.NA], dtype='string'),
    'tickets_count': pd.Series([0, 2, pd.NA, 1], dtype='Int64')
})
contact_filled = contact_example.assign(channel=contact_example['channel'].fillna('не указан'))
identified_contacts = contact_filled.dropna(subset=['customer_id'])
display(contact_filled)
print('Строк с известным покупателем:', len(identified_contacts))
print('Неизвестных чисел обращений осталось:', int(contact_filled['tickets_count'].isna().sum()))

,customer_id,channel,tickets_count
0,1,сайт,0
1,2,не указан,2
2,<NA>,телефон,<NA>
3,4,не указан,1


Строк с известным покупателем: 3
Неизвестных чисел обращений осталось: 1


Каждое правило возвращает отдельную булеву маску. Это позволяет одновременно увидеть несколько причин брака одной строки. Общая маска получается операцией `any(axis=1)` по нарушениям; число проблем по столбцам нельзя складывать как число уникальных отклонённых строк.

Проверяется наличие обязательных значений, положительность, равенство суммы произведению количества и цены, допустимый статус, существование связанных сущностей и конфликт ключа. Пропущенное количество отмечено самостоятельным правилом; неопределённость результата арифметики не должна незаметно удалять строку из контроля.

In [18]:
# Шаг 01.18 — Явные правила качества
issues = pd.DataFrame({
    'нет количества': working['qty'].isna(),
    'количество неположительно': working['qty'].le(0).fillna(False),
    'цена неположительна': working['price'].le(0),
    'сумма не согласована': working['amount'].ne(working['qty'] * working['price']).fillna(False),
    'нет даты': working['order_ts'].isna(),
    'неизвестный статус': ~working['status'].isin(status_names),
    'неизвестный город': working['city'].isna(),
    'нет покупателя': ~working['customer_id'].isin(customers['customer_id']),
    'нет товара': ~working['product_id'].isin(products['product_id']),
    'конфликт order_id': working.duplicated('order_id', keep=False)
}, index=working.index).fillna(False).astype(bool)
display(issues.sum().sort_values(ascending=False).to_frame('строк с нарушением'))

,строк с нарушением
сумма не согласована,3
конфликт order_id,2
количество неположительно,1
нет количества,1
нет даты,1
цена неположительна,1
неизвестный статус,1
нет покупателя,1
нет товара,1
неизвестный город,0


Строки разделяются на прошедшие проверки и отклонённые. К отклонённым добавляется текст причин; исходные значения сохраняются для разбора. Выбор «оставить первую запись» при конфликте идентификатора не используется, поскольку порядок файла не доказывает истинность записи.

В этом учебном договоре строки с неизвестными связями и обязательными пропусками не участвуют в аналитике до выяснения. В другой задаче допустима отдельная категория неизвестного справочника, но такое решение должно быть частью определения показателей. Объём исключений всегда показывается явно.

In [19]:
# Шаг 01.19 — Карантин с объяснением причин
bad_rows = issues.any(axis=1)
reason_text = issues.loc[bad_rows].apply(lambda row: '; '.join(row.index[row]), axis=1)
quarantine = working.loc[bad_rows].copy()
quarantine['причины'] = reason_text
clean_sample = working.loc[~bad_rows].copy()
display(quarantine[['order_id', 'qty', 'price', 'status', 'причины']])
display(pd.DataFrame({'состояние': ['вход', 'после точных дублей', 'карантин', 'принято'],
                      'строк': [len(dirty), len(working), len(quarantine), len(clean_sample)]}))

,order_id,qty,price,status,причины
0,1,<NA>,9700,delivered,нет количества
1,2,1,-100,delivered,цена неположительна; сумма не согласована
2,3,1,7630,delivered,сумма не согласована
5,6,1,3530,delivered,нет покупателя
6,7,1,5350,delivered,нет даты
7,8,0,119270,delivered,количество неположительно; сумма не согласована
8,9,3,2940,unknown,неизвестный статус
9,10,2,33170,delivered,нет товара
10,12,1,87190,delivered,конфликт order_id
11,12,1,1830,delivered,конфликт order_id


,состояние,строк
0,вход,21
1,после точных дублей,20
2,карантин,10
3,принято,10


`assert` фиксирует условия, которые должны выполняться после обработки. `pd.testing.assert_frame_equal` сравнивает значения, индекс, столбцы и типы двух таблиц. Это локальные проверки результата учебного преобразования, а не замена содержательной проверки бизнес-правил.

Проверки не доказывают универсальность очистки на любых входных данных. Они подтверждают, что на известном наборе дефектов выполнен конкретный договор. Конфликтные записи не исчезли без следа, принятые строки удовлетворяют условиям, а исходный фрагмент не изменён.

In [20]:
# Шаг 01.20 — Проверка результата и сохранности источника
assert len(working) == len(clean_sample) + len(quarantine)
assert clean_sample['order_id'].is_unique
assert clean_sample[['qty', 'order_ts']].notna().all().all()
assert clean_sample['qty'].gt(0).all() and clean_sample['price'].gt(0).all()
assert clean_sample['amount'].eq(clean_sample['qty'] * clean_sample['price']).all()
pd.testing.assert_frame_equal(orders.head(20), source_head)
print('Контроль пройден: принятые строки согласованы, карантин учтён, исходная таблица не изменена.')

Контроль пройден: принятые строки согласованы, карантин учтён, исходная таблица не изменена.


**Промежуточный результат.** Известные текстовые варианты нормализованы, точный повтор удалён, неоднозначные и некорректные строки отделены с причинами. Исходные таблицы не затронуты. На полном исходном наборе далее выполняется обогащение справочниками.

## 4. Соединения и контроль отношений между таблицами


Соединение сопоставляет строки по ключу. При связи «многие к одному» множество заказов ссылается на одну запись справочника, и левое соединение должно сохранять число заказов. Параметр `validate` превращает ожидаемую кардинальность в исполняемую проверку.

Одноимённые поля не всегда обозначают одно и то же. Город заказа и город покупателя могут различаться; цена в заказе и текущая цена каталога также имеют разный смысл. Имена столбцов после соединения должны отражать происхождение, чтобы дальнейшие расчёты не зависели от случайного суффикса.

Перед соединением город заказа переименован в `order_city`, город покупателя — в `customer_city`. Левое соединение оставляет каждый заказ, даже если запись справочника отсутствует. `indicator=True` добавляет источник совпадения; отсутствие пары видно как `left_only`.

Пустой правый столбец не всегда достаточен для диагностики: он может быть пустым и в существующей записи справочника. Индикатор проверяет именно факт сопоставления ключа. В полном наборе все покупатели находятся, но сохранение проверки защищает определение операции при новых данных.

In [21]:
# Шаг 01.21 — Обогащение заказов данными покупателей
orders_named = orders.rename(columns={'city': 'order_city', 'category': 'order_category', 'price': 'order_price'})
customers_named = customers.rename(columns={'city': 'customer_city'})
with_customers = orders_named.merge(customers_named, on='customer_id', how='left',
                                    validate='many_to_one', indicator=True)
display(with_customers[['order_id', 'customer_id', 'order_city', 'customer_city', 'segment', '_merge']].head(8))
display(with_customers['_merge'].value_counts().to_frame('заказов'))
assert len(with_customers) == len(orders)

,order_id,customer_id,order_city,customer_city,segment,_merge
0,1,8384,Санкт-Петербург,Санкт-Петербург,regular,both
1,2,11971,Краснодар,Краснодар,regular,both
2,3,19274,Новосибирск,Новосибирск,new,both
3,4,14821,Москва,Москва,new,both
4,5,6548,Москва,Москва,regular,both
5,6,17140,Новосибирск,Новосибирск,regular,both
6,7,18461,Нижний Новгород,Нижний Новгород,regular,both
7,8,6548,Москва,Москва,regular,both


,заказов
_merge,
both,200000
left_only,0
right_only,0


У каталога отдельно названы `catalog_category` и `catalog_price`, у заказа уже есть собственные `order_category` и `order_price`. В учебном наборе они согласованы; в исторических данных отличие цены может быть нормальным следствием изменения каталога.

Одновременная проверка количества строк и суммы обнаруживает разные классы нарушений. Совпадение одной общей суммы не доказывает корректность всех строк: ошибки могут взаимно компенсироваться. Поэтому проверка кардинальности, ключей и отдельных полей применяется совместно.

In [22]:
# Шаг 01.22 — Присоединение каталога с явными именами
products_named = products.rename(columns={'category': 'catalog_category', 'price': 'catalog_price'})
enriched = (with_customers.drop(columns='_merge')
            .merge(products_named, on='product_id', how='left', validate='many_to_one', indicator=True))
display(enriched[['order_id', 'product_id', 'name', 'order_category', 'catalog_category',
                  'order_price', 'catalog_price']].head(6))
print('Несовпадений категории:', int(enriched['order_category'].ne(enriched['catalog_category']).sum()))
print('Несовпадений цены:', int(enriched['order_price'].ne(enriched['catalog_price']).sum()))
assert len(enriched) == len(orders)
assert enriched['amount'].sum() == orders['amount'].sum()
assert enriched['_merge'].eq('both').all()

,order_id,product_id,name,order_category,catalog_category,order_price,catalog_price
0,1,1104,Одежда товар 1104,Одежда,Одежда,9700,9700
1,2,1471,Продукты товар 1471,Продукты,Продукты,480,480
2,3,887,Обувь товар 887,Обувь,Обувь,7630,7630
3,4,79,Одежда товар 79,Одежда,Одежда,11660,11660
4,5,1397,Бытовая техника товар 1397,Бытовая техника,Бытовая техника,57730,57730
5,6,960,Одежда товар 960,Одежда,Одежда,3530,3530


Несовпадений категории: 0
Несовпадений цены: 0


На маленьком примере виден состав результата каждого соединения. `inner` оставляет совпадающие ключи, `left` — все ключи слева, `right` — справа, `outer` — объединение. Пропуски в добавленных полях обозначают отсутствие стороны, а не нулевое значение.

Выбор типа соединения определяется вопросом. Для обогащения заказов обычно важно сохранить все заказы и отдельно проверить справочники. Внутреннее соединение может скрыть проблему, молча удалив записи с неизвестным покупателем. Синтаксически успешная операция не означает сохранения нужной генеральной совокупности.

In [23]:
# Шаг 01.23 — Различие inner left right и outer
left_demo = pd.DataFrame({'key': [1, 2], 'заказ': ['А', 'Б']})
right_demo = pd.DataFrame({'key': [2, 3], 'справочник': ['В', 'Г']})
join_examples = pd.concat({kind: left_demo.merge(right_demo, on='key', how=kind)
                           for kind in ['inner', 'left', 'right', 'outer']}, names=['соединение', 'строка'])
display(join_examples)

key заказ справочник
соединение строка                      
inner      0         2     Б          В
left       0         1     А        NaN
           1         2     Б          В
right      0         2     Б          В
           1         3   NaN          Г
outer      0         1     А        NaN
           1         2     Б          В
           2         3   NaN          Г

Если один ключ встречается слева два раза, а справа три, соединение создаёт шесть комбинаций. Это математическое следствие сопоставления, а не ошибка библиотеки. Неожиданное размножение строк искажает суммы и частоты.

В примере справочник намеренно содержит две версии товара. Сначала показан результат без проверки, затем ожидаемое исключение `MergeError` перехвачено и выведено как часть демонстрации. Исправлять такую ситуацию произвольным `drop_duplicates` без правила выбора версии нельзя.

In [24]:
# Шаг 01.24 — Размножение строк и validate
small_orders = pd.DataFrame({'order_id': [1, 2], 'product_id': [10, 10], 'amount': [100, 200]})
duplicate_catalog = pd.DataFrame({'product_id': [10, 10], 'version': ['старая', 'новая']})
multiplied = small_orders.merge(duplicate_catalog, on='product_id', how='left')
display(multiplied)
print('Сумма до:', small_orders['amount'].sum(), 'после:', multiplied['amount'].sum())
try:
    small_orders.merge(duplicate_catalog, on='product_id', validate='many_to_one')
except pd.errors.MergeError as error:
    print('Проверка остановила некорректную связь:', error)

,order_id,product_id,amount,version
0,1,10,100,старая
1,1,10,100,новая
2,2,10,200,старая
3,2,10,200,новая


Сумма до: 300 после: 600
Проверка остановила некорректную связь: Merge keys are not unique in right dataset; not a many-to-one merge


Левое соединение с `indicator` позволяет получить аналог anti join: записи слева, для которых нет ключа справа. Для проверки достаточно таблицы ключей справочника; остальные столбцы не нужны.

Поиск неизвестных связей выполняется на учебной копии, где есть специально введённый покупатель `999999`. В полном исходном наборе таких случаев нет. Диагностика должна уметь показывать и нулевой результат, не создавая вымышленных проблем в чистых данных.

In [25]:
# Шаг 01.25 — Поиск отсутствующих связей
relation_check = working[['order_id', 'customer_id']].merge(
    customers[['customer_id']], on='customer_id', how='left', validate='many_to_one', indicator=True)
missing_customers = relation_check.loc[relation_check['_merge'].eq('left_only'), ['order_id', 'customer_id']]
display(missing_customers)
print('Заказов без покупателя в учебной копии:', len(missing_customers))

,order_id,customer_id
5,6,999999


Заказов без покупателя в учебной копии: 1


`concat` добавляет таблицы друг к другу по оси, а не ищет пары по бизнес-ключу. При добавлении строк столбцы выравниваются по именам; отсутствующие поля становятся пустыми. `ignore_index=True` создаёт новый последовательный индекс, но не устраняет повторяющиеся бизнес-идентификаторы.

При объединении выгрузок важна общая схема. Наличие нового столбца только в одной части не означает, что он заполнен в остальных периодах. Контроль количества строк и уникальности ключа после `concat` остаётся отдельной обязанностью.

In [26]:
# Шаг 01.26 — Объединение частей concat и выравнивание полей
part_a = orders.loc[:2, ['order_id', 'amount']].copy()
part_b = orders.loc[3:5, ['order_id', 'amount', 'status']].copy()
combined_parts = pd.concat([part_a, part_b], ignore_index=True, sort=False)
display(combined_parts)
print('Бизнес-ключ уникален:', combined_parts['order_id'].is_unique)

,order_id,amount,status
0,1,9700,NaN
1,2,480,NaN
2,3,7630,NaN
3,4,11660,returned
4,5,57730,delivered
5,6,3530,delivered


Бизнес-ключ уникален: True


**Промежуточный результат.** Полные заказы обогащены справочниками без потери и размножения строк. Происхождение цен, категорий и городов отражено в именах. На малых примерах проверены типы соединений, отсутствие связей и нарушение кардинальности.

## 5. Связная задача подготовки аналитической таблицы


Финальная задача объединяет выборку, календарные признаки и контроль качества. Она не требует повторной загрузки данных: используются объекты, уже созданные в последовательности. Задачи сопровождаются выполненными решениями, чтобы результат можно было сопоставить с объяснением.

Таблица готовится для анализа заказов, а не для машинного обучения. Правила временного разделения и конструирование признаков для прогнозирования будут отдельной темой курса. Здесь вычисляются только характеристики, необходимые для описания имеющихся записей.

**Задача.** Получить десять крупнейших возвращённых заказов за второе полугодие с количеством не меньше двух. В результате нужны идентификатор заказа, дата, товар, категория, город заказа и сумма. При равных суммах порядок определяется идентификатором.

Сначала задаётся период полуинтервалом: начало включено, конец исключён. Такой способ надёжно охватывает все часы и секунды последнего дня. Затем применяются статус и количество. Проверки относятся к полной найденной выборке, а десять строк используются только для представления.

In [27]:
# Шаг 01.27 — Задача выборки с несколькими условиями
task_mask = (enriched['order_ts'].ge(pd.Timestamp('2026-07-01'))
             & enriched['order_ts'].lt(pd.Timestamp('2027-01-01'))
             & enriched['status'].eq('returned')
             & enriched['qty'].ge(2))
task_result = (enriched.loc[task_mask, ['order_id', 'order_ts', 'name', 'order_category', 'order_city', 'qty', 'amount']]
               .sort_values(['amount', 'order_id'], ascending=[False, True]))
display(task_result.head(10))
assert task_result['order_ts'].ge('2026-07-01').all()
assert task_result['qty'].ge(2).all()
print('Подходящих заказов:', len(task_result))

,order_id,order_ts,name,order_category,order_city,qty,amount
114423,114424,2026-07-03 13:29:59,Электроника товар 888,Электроника,Новосибирск,5,577950
143151,143152,2026-07-21 02:48:13,Электроника товар 1882,Электроника,Ростов-на-Дону,5,559100
173971,173972,2026-08-15 02:10:28,Электроника товар 1419,Электроника,Москва,5,539250
125450,125451,2026-08-19 13:13:47,Электроника товар 1430,Электроника,Самара,5,530600
7311,7312,2026-12-21 16:33:59,Электроника товар 761,Электроника,Самара,5,515450
124823,124824,2026-09-19 04:53:36,Электроника товар 1718,Электроника,Москва,4,451800
82224,82225,2026-12-29 18:32:54,Электроника товар 1144,Электроника,Москва,4,449640
155698,155699,2026-08-21 21:09:29,Электроника товар 1713,Электроника,Санкт-Петербург,5,425200
131546,131547,2026-08-31 13:30:27,Электроника товар 1430,Электроника,Омск,4,424480
72055,72056,2026-07-22 14:15:33,Электроника товар 1721,Электроника,Москва,4,400520


Подходящих заказов: 1347


Регистрация позже заказа нарушала бы предполагаемую хронологию клиента. Разница между городом покупателя и городом заказа сама по себе не является дефектом: нужна информация о значении этих атрибутов. Проверка должна отделять формальное отличие от нарушения договора данных.

Возраст регистрации считается на календарные даты: время заказа нормализуется к началу дня. Это даёт число календарных дней между датами. При расчёте полных 24-часовых интервалов правило было бы другим. Выбор единицы и округления является частью определения признака.

In [28]:
# Шаг 01.28 — Согласованность дат и различие городов
enriched['registration_age_days'] = (enriched['order_ts'].dt.normalize() - enriched['registered']).dt.days
consistency = pd.DataFrame({
    'наблюдение': ['регистрация позже даты заказа', 'город заказа отличается от города клиента',
                   'категория заказа отличается от каталога'],
    'строк': [int(enriched['registration_age_days'].lt(0).sum()),
              int(enriched['order_city'].ne(enriched['customer_city']).sum()),
              int(enriched['order_category'].ne(enriched['catalog_category']).sum())]
})
display(consistency)
display(enriched[['order_id', 'registered', 'order_ts', 'registration_age_days']].head(5))

,наблюдение,строк
0,регистрация позже даты заказа,0
1,город заказа отличается от города клиента,0
2,категория заказа отличается от каталога,0


,order_id,registered,order_ts,registration_age_days
0,1,2022-01-17,2026-01-17 08:50:04,1461
1,2,2023-07-06,2026-07-13 14:51:05,1103
2,3,2025-03-29,2026-11-03 13:42:32,584
3,4,2024-02-28,2026-03-19 16:38:33,750
4,5,2022-10-17,2026-11-27 01:54:49,1502


Паспорт объединяет ключевые гарантии подготовки: уникальность заказа, полноту связей, арифметику, сохранность суммы и согласованность хронологии. Значение `True` означает выполнение конкретного правила на текущем наборе, а не универсальную достоверность всех сведений.

Финальная таблица сохраняет исходные поля и происхождение атрибутов. Денежный показатель `delivered_amount` имеет явное определение и не подменяет бухгалтерскую выручку. Сохранение файловых форматов и проверка Parquet рассматриваются в четвёртом ноутбуке, поэтому здесь результат остаётся объектом в памяти.

In [29]:
# Шаг 01.29 — Паспорт готовности и итоговая таблица
prepared = enriched.drop(columns='_merge').copy()
quality_passport = pd.Series({
    'order_id уникален': prepared['order_id'].is_unique,
    'число заказов сохранено': len(prepared) == len(orders),
    'все покупатели найдены': prepared['customer_id'].isin(customers['customer_id']).all(),
    'все товары найдены': prepared['product_id'].isin(products['product_id']).all(),
    'сумма не изменилась': prepared['amount'].sum() == orders['amount'].sum(),
    'арифметика суммы согласована': prepared['amount'].eq(prepared['qty'] * prepared['order_price']).all(),
    'давность регистрации неотрицательна': prepared['registration_age_days'].ge(0).all()
}, name='выполнено')
display(quality_passport.to_frame())
assert quality_passport.all()
display(prepared[['order_id', 'order_ts', 'customer_id', 'product_id', 'name', 'order_category',
                  'qty', 'order_price', 'amount', 'status', 'segment', 'registration_age_days']].head(6))
print('Подготовлено заказов:', len(prepared))
print('Стоимость доставленных заказов:', int(prepared['delivered_amount'].sum()))

,выполнено
order_id уникален,True
число заказов сохранено,True
все покупатели найдены,True
все товары найдены,True
сумма не изменилась,True
арифметика суммы согласована,True
давность регистрации неотрицательна,True


,order_id,order_ts,customer_id,product_id,name,order_category,qty,order_price,amount,status,segment,registration_age_days
0,1,2026-01-17 08:50:04,8384,1104,Одежда товар 1104,Одежда,1,9700,9700,delivered,regular,1461
1,2,2026-07-13 14:51:05,11971,1471,Продукты товар 1471,Продукты,1,480,480,delivered,regular,1103
2,3,2026-11-03 13:42:32,19274,887,Обувь товар 887,Обувь,1,7630,7630,delivered,new,584
3,4,2026-03-19 16:38:33,14821,79,Одежда товар 79,Одежда,1,11660,11660,returned,new,750
4,5,2026-11-27 01:54:49,6548,1397,Бытовая техника товар 1397,Бытовая техника,1,57730,57730,delivered,regular,1502
5,6,2026-07-20 15:42:07,17140,960,Одежда товар 960,Одежда,1,3530,3530,delivered,regular,898


Подготовлено заказов: 200000
Стоимость доставленных заказов: 5135476730


**Итог.** Построена таблица заказов с данными покупателей и товаров. Проверены ключи, арифметика, хронология и сохранение строк. Практика очистки выполнена на отдельной копии с сохранением причин отклонения.

Следующий ноутбук использует ту же предметную область для группировок, временных рядов, RFM и когортного анализа. Основной принцип подготовки сохраняется: результат операции оценивается по смыслу, а не только по отсутствию исключений.